In [27]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os
import shutil

pdf_path = "data/피지컬_AI_현황과_시사점.pdf"

# 영문 파일명으로 변경
new_path = "data/physical_ai_report.pdf"

if os.path.exists(pdf_path):
    shutil.copy2(pdf_path, new_path)
else:
    print(f"파일이 존재하지 않습니다: {pdf_path}")

# 페이지별로 이미지로 만듬.

In [ ]:
from pathlib import Path

from docling_core.types.doc import ImageRefMode, PictureItem, TableItem
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption

# 해상도
IMAGE_RESOLUTION_SCALE = 2.0

input_doc_path = Path("data/physical_ai_report.pdf")
output_dir = Path("Scretch2")

pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE
pipeline_options.generate_page_images = True # 페이지 이미지 생성 활성화
# pipeline_options.generate_picture_images = True # 페이지 안에 있는 이미지 값들만 vlm을 통해서 캡셔닝

# 문서 변환
doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options) #PDF파일의 형식을 넣을것이다.
    }
)

# PDF 문서 변환 실행
conv_res = doc_converter.convert(input_doc_path)

c:\Users\skyop\jaeho_template\venv_docling\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-24 18:13:05,477 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-10-24 18:13:05,552 - INFO - Going to convert document batch...
2025-10-24 18:13:05,552 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 8b7da28f2b5f716d6ceef85a126f3582
2025-10-24 18:13:05,567 - INFO - Loading plugin 'docling_defaults'
2025-10-24 18:13:05,567 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-24 18:13:05,581 - INFO - Loading plugin 'docling_defaults'
2025-10-24 18:13:05,588 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-24 18:13:05,605 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-10-24 18:13:05,723

# 이미지로 변환된 문서 저장

In [5]:
import logging
import time

logging.basicConfig(level=logging.INFO)
_log = logging.getLogger(__name__)

start_time = time.time()

# 출력 디렉토리 생성 (부모 디렉토리도 함께 생성)
output_dir.mkdir(parents=True, exist_ok=True)
doc_filename = conv_res.input.file.stem

# 페이지 이미지 저장
for page_no, page in conv_res.document.pages.items():
    page_no = page.page_no
    page_image_filename = output_dir / f"{doc_filename}-{page_no}.png"
    with page_image_filename.open("wb") as fp:
        page.image.pil_image.save(fp, format="PNG")

# 변환 완료 시간 계산
end_time = time.time() - start_time

# 변환 완료 로그 출력
_log.info(f"Document converted and exported in {end_time:.2f} seconds.")

2025-10-24 18:14:41,941 - INFO - Document converted and exported in 2.60 seconds.


# 페이지별 텍스트 추출 빛 이미지 매칭
변환된 문서에서 페이지별로 텍스트를 추출하고, 해당하는 페이지 이미지와 매칭하여, 멀티모달 입력 데이터를 준비

In [8]:
from itertools import accumulate
result = conv_res
page_element_count = [1+len(i.assembled.elements) for i in result.pages]
element_page_cutoff = list(accumulate([1]+page_element_count))

pages = [
    (
        result.document.export_to_markdown(
            from_element=s,
            to_element=e
        )
    )
for s, e in zip (element_page_cutoff[:-1], element_page_cutoff[1:])
]


In [10]:
print(element_page_cutoff,"\n", page_element_count)

[1, 8, 13, 16, 21, 26, 43, 61, 76, 95, 110, 128, 148, 164, 180, 196, 209, 225, 245, 261, 279, 293, 307, 319, 338, 356, 375, 389, 408, 424, 438, 454, 471, 488, 507, 522, 537, 557, 574, 590, 597, 614, 632, 647, 681, 714, 740, 745, 752] 
 [7, 5, 3, 5, 5, 17, 18, 15, 19, 15, 18, 20, 16, 16, 16, 13, 16, 20, 16, 18, 14, 14, 12, 19, 18, 19, 14, 19, 16, 14, 16, 17, 17, 19, 15, 15, 20, 17, 16, 7, 17, 18, 15, 34, 33, 26, 5, 7]


In [13]:
pages

['ISSUE REPORT l 2025.05.13. IS-202\n\n## 피지컬AI의현황과시사점\n\nCurrentStatusandImplicationsofPhysicalAI\n\n이해수,유재흥,안성원\n\n<!-- image -->\n\n<!-- image -->',
 'SPRi이슈리포트IS-202\n\n피지컬AI의현황과시사점\n\n이보고서는 ｢ 과학기술정보통신부정보통신진흥기금 ｣ 에서지원받아제작한것으로 과학기술정보통신부의공식의견과다를수있습니다. 이보고서의내용은연구진의개인견해이며,본보고서와관련한의문사항또는수정·보완할 필요가있는경우에는아래연락처로연락해주시기바랍니다.',
 '소프트웨어정책연구소AI정책연구실 이해수선임연구원(hs.lee@spri.kr)\n\n## CONTENT\n\n| Ⅰ.                                                                                                   | 피지컬AI의 부상···········································································1          |                                                                               |\n|------------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|\n| Ⅱ.                                       

# 멀티모달 임베딩을 위한 텍스트-이미징 페어링

In [24]:
import os

texts = []
image_paths = []

# 각 페이지의 텍스트를 texts list에 추가
for page_text in pages:
    texts.append(page_text)

# Scratch2 폴더에서 physical_ai_report={page_num}.png 이미지 파일들 수집
scratch2_dir = "./Scratch2"
for i in range(len(pages)):
    img_path = os.path.join(scratch2_dir, f"physical_ai_report-{i+1}.png")
    if os.path.exists(img_path):
        image_paths.append(img_path)

# zip함수로 합쳐서 inputs에 저장
from PIL import Image

inputs = []
for text, image_path in zip(texts, image_paths):
    try: 
        image = Image.open(image_path)
        inputs.append([text, image])
    
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")

In [25]:
inputs

[['ISSUE REPORT l 2025.05.13. IS-202\n\n## 피지컬AI의현황과시사점\n\nCurrentStatusandImplicationsofPhysicalAI\n\n이해수,유재흥,안성원\n\n<!-- image -->\n\n<!-- image -->',
  <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1190x1682>],
 ['SPRi이슈리포트IS-202\n\n피지컬AI의현황과시사점\n\n이보고서는 ｢ 과학기술정보통신부정보통신진흥기금 ｣ 에서지원받아제작한것으로 과학기술정보통신부의공식의견과다를수있습니다. 이보고서의내용은연구진의개인견해이며,본보고서와관련한의문사항또는수정·보완할 필요가있는경우에는아래연락처로연락해주시기바랍니다.',
  <PIL.PngImagePlugin.PngImageFile image mode=RGB size=1190x1682>],
 ['소프트웨어정책연구소AI정책연구실 이해수선임연구원(hs.lee@spri.kr)\n\n## CONTENT\n\n| Ⅰ.                                                                                                   | 피지컬AI의 부상···········································································1          |                                                                               |\n|------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------

# Voyage AI의 임베딩 API를 활용하기 위햐 API키를 불러오고, 텍스트-이미지 데이터를 임베딩.

In [ ]:
import voyageai

voyageai_client = voyageai.Client()

result = voyageai_client.multimodal_embed(
    inputs=inputs,
    model="voyage-multimodal-3",
    input_type="document" # query 와 document 2가지
)

embeddings = result.embeddings
print(f"총 벡터 개수: {len(embeddings)}")

총 벡터 개수: 48


# 허깅페이스 모델 jina-clip-v2모델 사용해보기

https://huggingface.co/jinaai/jina-clip-v2

In [ ]:
# # !pip install transformers torch einops timm pillow

# from transformers import AutoModel, AutoProcessor
# import torch

# # 모델과 프로세서 로드
# model = AutoModel.from_pretrained('jinaai/jina-clip-v2', trust_remote_code=True)
# processor = AutoProcessor.from_pretrained('jinaai/jina-clip-v2', trust_remote_code=True)

# # inputs는 이미 (text, image) 쌍의 리스트로 zip을 활용해 만들어져 있음
# # 각 쌍에 대해 processor를 통해 텍스트와 이미지를 동일 인덱스 기준으로 처리
# texts_for_jina = [text for text, image in inputs]
# images_for_jina = [image for text, image in inputs]

# # processor가 텍스트/이미지 쌍을 받을 수 있도록 각각 리스트로 전달
# processed = processor(
#     text=texts_for_jina,
#     images=images_for_jina,
#     return_tensors="pt",
#     padding=True
# )

# # 임베딩 추출
# with torch.no_grad():
#     outputs = model(**processed)
#     # 예시: outputs.pooled_output 등 사용, 실제 모델 아웃풋 명칭 확인 필요
#     jina_clip_embeddings = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs.last_hidden_state



2025-10-25 00:12:32,496 - INFO - `text_config` is `None`. Initializing the `JinaCLIPTextConfig` with default values.
2025-10-25 00:12:32,497 - INFO - `vision_config` is `None`. initializing the `JinaCLIPVisionConfig` with default values.


# Qdrant 벡터 데이터 베이스에 임베딩 저장
벡터는 1024차원, 코사인 유사도 사용

In [39]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid


qdrant_client = QdrantClient("localhost", port=6333, grpc_port=True)
collection_name = "Practice_Multimodal_retriever"

# qdrant_client.get_collection(collection_name)
try:
    qdrant_client.get_collection(collection_name)
    print(f"컬렉션 {collection_name}이 이미 존재합니다.")
except Exception as e:
    # 컬렉션 생성
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=1024, distance=models.Distance.COSINE),
        on_disk_payload=True # 메타데이터를 디스크에 저장하여 메모리 효율성 향상
        )
    print(f"컬렉션 {collection_name}이 생성되었습니다.")


    # Qdrant에 문서 벡터 업로드
    points = []
    for i, vector in enumerate(embeddings):
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vector,
                payload={
                    "source": "voyage", # 임베딩 생성 정보
                    "text": texts[i], # 해당 텍스트의 내용
                    "image_path": image_paths[i]
                    
                }
            )
        )

    # 생성된 모든 포인트를 Qdrant 컬렉션에 업로드
    qdrant_client.upsert(collection_name=collection_name, points=points)
    print(f"총 {len(points)}개의 벡터가 Qdrant 컬렉션에 업로드되었습니다.")



2025-10-24 23:47:29,873 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2025-10-24 23:47:29,902 - INFO - HTTP Request: GET http://localhost:6333/collections/Practice_Multimodal_retriever "HTTP/1.1 404 Not Found"
2025-10-24 23:47:30,081 - INFO - HTTP Request: PUT http://localhost:6333/collections/Practice_Multimodal_retriever "HTTP/1.1 200 OK"
2025-10-24 23:47:30,193 - INFO - HTTP Request: PUT http://localhost:6333/collections/Practice_Multimodal_retriever/points?wait=true "HTTP/1.1 200 OK"


컬렉션 Practice_Multimodal_retriever이 생성되었습니다.
총 48개의 벡터가 Qdrant 컬렉션에 업로드되었습니다.


# 멀티모달 검색기(Multimodal Retriever)구현

In [8]:
from typing import List
from langchain_core.documents import Document
import voyageai

from qdrant_client import QdrantClient
from qdrant_client.http import models
import uuid


qdrant_client = QdrantClient("localhost", port=6333, grpc_port=True)
collection_name = "Practice_Multimodal_retriever"
voyageai_client = voyageai.Client()


class MultimodalRetriever:
    def __init__(self, qdrant_client, collection_name:str, votageai_client, limit: int = 5):
        self.qdrant_client = qdrant_client
        self.collection_name = collection_name
        self.voyageai_client = votageai_client
        self.limit = limit # 검색해서 뽑아낼 문서의 개수

    def invoke(self, query:str) -> List[Document]:
        """질의에 대해 관련 문서와 이미지를 검색하여 Document 형태로 변환"""
        query_result = self.voyageai_client.multimodal_embed(
            inputs=[query],
            model="voyage-multimodal-3",
            input_type="query" # 사용자의 질문이기 때문에
        )        
        query_vector = query_result.embeddings[0] #query vector를 임베딩의 첫번쨰 값으로 저장

        # Qdrant에서 검색
        search_result = self.qdrant_client.search(
            collection_name=self.collection_name,
            query_vector=query_vector,
            limit=self.limit
        )

        documents = []
        
        # langchain의 Document 형식으로 변환
        for hit in search_result:
            doc = Document(
                page_content=hit.payload.get("text", ""), # 이미지 내용
                metadata={
                    'score': hit.score, # 유사도 점수
                    'image_path': hit.payload.get('image_path', ''),
                    **hit.payload
                }
            )
            documents.append(doc)
        return documents       

In [9]:
retriever = MultimodalRetriever(
    qdrant_client=qdrant_client,
    collection_name=collection_name,
    votageai_client=voyageai_client,
    limit=5
)

query = ["회사 로고 많은 페이지"]

retriever.invoke(query)

C:\Users\skyop\AppData\Local\Temp\ipykernel_3392\1206930017.py:32: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  search_result = self.qdrant_client.search(


[Document(metadata={'score': 0.23337837, 'image_path': './Scratch2\\physical_ai_report-47.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.228598, 'image_path': './Scratch2\\physical_ai_report-48.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.20137338, 'image_path': './Scratch2\\physical_ai_report-16.png', 'source': 'voyage', 'text': ''}, page_content=''),
 Document(metadata={'score': 0.19793469, 'image_path': './Scratch2\\physical_ai_report-26.png', 'source': 'voyage', 'text': "<!-- image -->\n\n* 출처 : https://www.figure.ai/\n\n- 2025년 2월, 자사 로봇의 인식, 언어 이해, 제어 기능을 통합하여 기존 로봇 공학의 한계를 극복하기 위해 시각-언어-행동(VLA, Vision-Language-Action) 모델 '헬릭스(Helix)'를 공개(Figure.ai, 2025. 2.)\n- 기존에는 로봇에게 새로운 행동을 가르치려면 전문가의 수동 프로그래밍이나 수천 번의 시뮬레이션이 필요했으나, 헬릭스는 로봇이 카메라로 수집한 시각 정보와 자연어 명령을 결합하여 이전에 접한 적 없는 물체도 실시간으로 조작\n- 작업별 미세조정 없이 단일 신경망으로 모든 동작을 학습하는 한편, 기기 내장형 저전력 GPU에서 실행 할 수 있어 신속한 상용화를 지원\n- 시스템 2(S2)와 시스템 1(S1)이라는 2개의 시스템을 통

# RAG 시스템 구현

이미지 값을 base64로 인코딩 하여야 llm에게 넘겨 줄수 있다는것을 명심!

In [10]:
import os
import base64

def encode_inage_to_base64(image_path):
    """이미지를 base64로 인코딩"""
    if os.path.exists(image_path):
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

ModuleNotFoundError: No module named 'langchain_openai'